In [ ]:
!pip install ultralytics open_clip_torch transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import random
import shutil

BASE_PATH = "/content/drive/MyDrive/copro/data"

RAW_PATH = os.path.join(BASE_PATH, "raw")
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH = os.path.join(BASE_PATH, "val")

classes = ["neem", "tulasi"]

for cls in classes:

    raw_class = os.path.join(RAW_PATH, cls)
    train_class = os.path.join(TRAIN_PATH, cls)
    val_class = os.path.join(VAL_PATH, cls)

    os.makedirs(train_class, exist_ok=True)
    os.makedirs(val_class, exist_ok=True)

    images = [
        f for f in os.listdir(raw_class)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(images)

    split_index = int(0.8 * len(images))

    train_images = images[:split_index]
    val_images = images[split_index:]

    # Copy training images
    for img in train_images:
        shutil.copy(
            os.path.join(raw_class, img),
            os.path.join(train_class, img)
        )

    # Copy validation images
    for img in val_images:
        shutil.copy(
            os.path.join(raw_class, img),
            os.path.join(val_class, img)
        )

    print(f"{cls} → Train: {len(train_images)} | Val: {len(val_images)}")

print("✅ 80:20 dataset split completed successfully")

In [ ]:
from ultralytics import YOLO
import torch

model = YOLO("yolov8m-cls.yaml")

model.train(
    data="/content/drive/MyDrive/copro/data",
    epochs=25,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu"
)

In [ ]:
import os
import json
import cv2
import numpy as np
from ultralytics import YOLO
import open_clip
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

In [ ]:
class QualityMetrics:

    def compute(self, image_path):

        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        blur = cv2.Laplacian(img, cv2.CV_64F).var()
        brightness = np.mean(img) / 255
        contrast = np.std(img) / 255

        return {
            "blur_score": round(float(blur),3),
            "brightness": round(float(brightness),3),
            "contrast": round(float(contrast),3)
        }

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

class YOLOExtractor:

    def __init__(self):

        # Your trained classification model
        self.model = YOLO("/content/runs/classify/train/weights/best.pt")

        self.conf = 0.05


    def extract(self, image_path):

        objects = []

        image = cv2.imread(image_path)

        if image is None:
            return objects

        h, w = image.shape[:2]
        image_area = h * w

        results = self.model.predict(
            source=image_path,
            conf=self.conf,
            imgsz=640,
            verbose=False
        )

        for r in results:

            if r.probs is None:
                continue

            class_id = int(r.probs.top1)
            confidence = float(r.probs.top1conf)

            label = self.model.names[class_id]

            # Whole image as bounding box
            x1, y1, x2, y2 = 0, 0, w, h

            bbox_area = w * h
            area_ratio = bbox_area / image_area

            # Calculate green ratio
            hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

            lower_green = np.array([35,40,40])
            upper_green = np.array([85,255,255])

            mask = cv2.inRange(hsv, lower_green, upper_green)

            green_pixels = np.sum(mask > 0)
            total_pixels = h * w

            green_ratio = green_pixels / total_pixels

            # Leaf stage estimation
            if green_ratio > 0.6:
                leaf_stage = "Mature"
            elif green_ratio > 0.3:
                leaf_stage = "Young"
            else:
                leaf_stage = "Old"

            objects.append({
                "label": label,
                "confidence": round(confidence,3),
                "bbox": [x1,y1,x2,y2],
                "area_ratio": round(float(area_ratio),3),
                "green_ratio": round(float(green_ratio),3),
                "leaf_stage": leaf_stage
            })

        return objects

In [ ]:
class CLIPExtractor:

    def __init__(self):

        self.model,_,self.preprocess = open_clip.create_model_and_transforms(
            "ViT-B-32", pretrained="openai"
        )

        self.model.eval()

    def extract(self,image_path):

        image = self.preprocess(Image.open(image_path)).unsqueeze(0)

        with torch.no_grad():
            embedding = self.model.encode_image(image)

        return embedding.cpu().numpy().flatten().tolist()

In [ ]:
class BLIPExtractor:

    def __init__(self):

        self.processor = BlipProcessor.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )

        self.model = BlipForConditionalGeneration.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )

    def extract(self,image_path):

        image = Image.open(image_path).convert("RGB")

        inputs = self.processor(image, return_tensors="pt")

        out = self.model.generate(**inputs)

        caption = self.processor.decode(out[0], skip_special_tokens=True)

        return caption

In [ ]:
class FeatureFusion:

    def fuse(self,image_name,quality,clip_emb,caption,objects):

        return {
            "image":image_name,

            "quality_metrics":quality,

            "semantic_features":{
                "clip_embedding":clip_emb
            },

            "descriptive_features":{
                "caption":caption
            },

            "object_features":{
                "detected_objects":objects,
                "object_count":len(objects)
            }
        }

In [ ]:
quality_checker = QualityMetrics()

yolo_extractor = YOLOExtractor()

clip_extractor = CLIPExtractor()

blip_extractor = BLIPExtractor()

fusion_engine = FeatureFusion()

In [ ]:
import os
import json

DATASET_DIR = "/content/drive/MyDrive/copro/data/raw"
OUTPUT_DIR = "/content/drive/MyDrive/copro/output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
final_dataset = []

for root, dirs, files in os.walk(DATASET_DIR):

    for img_name in files:

        if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_path = os.path.join(root, img_name)

        print("Processing:", img_name)

        quality = quality_checker.compute(image_path)

        clip_emb = clip_extractor.extract(image_path)

        caption = blip_extractor.extract(image_path)

        objects = yolo_extractor.extract(image_path)

        structured = fusion_engine.fuse(
            img_name,
            quality,
            clip_emb,
            caption,
            objects
        )

        final_dataset.append(structured)

In [ ]:
OUTPUT_JSON = os.path.join(
    OUTPUT_DIR,
    "plant_multimodal_dataset.json"
)

with open(OUTPUT_JSON, "w") as f:
    json.dump(final_dataset, f, indent=4)

print("JSON saved at:", OUTPUT_JSON)
print("Total images processed:", len(final_dataset))